# Configuração de ambiente

In [ ]:
import pandas as pd
import numpy as np
import time
import json
import multiprocessing as mp
from sklearn.datasets import load_breast_cancer

N_WORKERS = mp.cpu_count()
CHUNK_SIZE = None 

def load_data():
    data = load_breast_cancer(as_frame=True)
    df = data.frame.copy()
    df.columns = [c.replace(' ', '_').replace('(', '').replace(')', '') for c in df.columns]
    return df

df = load_data()
print("Colunas disponíveis:", df.columns.tolist()[:5]) 

Colunas disponíveis: ['mean_radius', 'mean_texture', 'mean_perimeter', 'mean_area', 'mean_smoothness']


# Fork-Join / Map-Reduce

In [ ]:
def map_stats(chunk):
    """
    EXECUTADO EM PARALELO (Workers)
    Retorna estatísticas locais de um pedaço do dataset.
    """
    numeric_df = chunk.select_dtypes(include=[np.number])
    
    return {
        'count': numeric_df.count(),
        'sum': numeric_df.sum(),
        'sum_sq': (numeric_df**2).sum(),
        'min': numeric_df.min(),
        'max': numeric_df.max(),
        'nan': numeric_df.isna().sum()
    }

def reduce_stats(local_results):
    """
    EXECUTADO EM SEQUENCIAL (Processo Pai)
    Consolida os dicionários locais em estatísticas globais.
    """
    res_global = local_results[0].copy()
    
    # Agregação (Reduce)
    for i in range(1, len(local_results)):
        res_global['count'] += local_results[i]['count']
        res_global['sum']   += local_results[i]['sum']
        res_global['sum_sq'] += local_results[i]['sum_sq']
        res_global['nan']   += local_results[i]['nan']
        # Min e Max globais
        res_global['min'] = np.minimum(res_global['min'], local_results[i]['min'])
        res_global['max'] = np.maximum(res_global['max'], local_results[i]['max'])
    
    # Média e Desvio Padrão Global
    n = res_global['count']
    mean_global = res_global['sum'] / n
    
    # Variância = (Soma_sq / n) - (Média^2)
    var_global = (res_global['sum_sq'] / n) - (mean_global**2)
    std_global = np.sqrt(var_global)
    
    return mean_global, std_global, res_global['min'], res_global['max']

# --- Execução do Teste ---
chunks = np.array_split(df, N_WORKERS)


local_stats = [map_stats(c) for c in chunks]
mean_g, std_g, min_g, max_g = reduce_stats(local_stats)

print("Estatísticas Globais (via Map-Reduce) calculadas!")
print(f"Média da primeira coluna: {mean_g.iloc[0]:.4f}")

Estatísticas Globais (via Map-Reduce) calculadas!
Média da primeira coluna: 14.1273


c:\Users\Luca\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


# Implementação do Pipeline
## Producer - Worker - Aggregator

In [ ]:
import multiprocessing as mp
import utils  
import importlib
importlib.reload(utils) 

if __name__ == "__main__":
    input_q = mp.Queue()
    output_q = mp.Queue()
    
    # --- PRODUCER ---
    chunks = np.array_split(df, N_WORKERS)
    print(f"[Producer] Criando {len(chunks)} chunks. Realizado.")
    for i, chunk in enumerate(chunks):
        input_q.put((i, chunk))
    for _ in range(N_WORKERS):
        input_q.put(None)
    print(f"[Producer] Enviado. Realizado.")

    # --- WORKERS ---
    workers = []
    for i in range(N_WORKERS):

        p = mp.Process(target=utils.worker_process, args=(input_q, output_q, i))
        p.start()
        workers.append(p)

    # --- AGGREGATOR ---
    results = []
    finished_workers = 0
    print(f"[Aggregator] Coletando...")
    
    while finished_workers < N_WORKERS:
        idx, res = output_q.get() 
        if idx is None:
            finished_workers += 1
        else:
            results.append((idx, res))
            print(f"Recebido chunk {idx}. Realizado.")


    errors = [r for r in results if isinstance(r[1], str)]
    if errors:
        print("\n[ERRO] Alguns chunks falharam:")
        for idx, err_msg in errors:
            print(f"  - Chunk {idx}: {err_msg}")
    else:
        results.sort(key=lambda x: x[0])
        
        # Extrair apenas os DataFrames
        df_list = [r[1] for r in results]
        
        X_preprocessed = pd.concat(df_list)
        print(f"\n[FIM] Sucesso! Shape final: {X_preprocessed.shape}")
        display(X_preprocessed.head())
    
    for p in workers:
        p.join()

    print(f"\n[FIM] Sucesso! Shape final: {X_preprocessed.shape}")

c:\Users\Luca\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


[Producer] Criando 8 chunks. Realizado.
[Producer] Enviado. Realizado.
[Aggregator] Coletando...
Recebido chunk 0. Realizado.
Recebido chunk 2. Realizado.
Recebido chunk 1. Realizado.
Recebido chunk 5. Realizado.
Recebido chunk 4. Realizado.
Recebido chunk 6. Realizado.
Recebido chunk 7. Realizado.
Recebido chunk 3. Realizado.

[FIM] Sucesso! Shape final: (569, 36)


,mean_radius,mean_texture,mean_perimeter,mean_area,mean_smoothness,mean_compactness,mean_concavity,mean_concave_points,mean_symmetry,mean_fractal_dimension,...,worst_concavity,worst_concave_points,worst_symmetry,worst_fractal_dimension,target,radius_x_texture,area_sq,concavity_per_smoothness,row_mean,row_std
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,0.7119,0.2654,0.4601,0.11890,0,186.7362,1002001.00,2.534628,29581.072038,171822.656759
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,0.2416,0.1860,0.2750,0.08902,0,365.5289,1758276.00,1.025490,51834.808172,301520.764187
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,0.4504,0.2430,0.3613,0.08758,0,418.4125,1447209.00,1.801095,42676.959004,248174.761707
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,0.6869,0.2575,0.6638,0.17300,0,232.7396,149073.21,1.694035,4427.871926,25558.465238
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,0.4000,0.1625,0.2364,0.07678,0,290.9586,1682209.00,1.974078,49583.488732,288477.805400



[FIM] Sucesso! Shape final: (569, 36)


# Brenchmarking 
## Exportação

In [ ]:
import time
import json

def run_sequential(df):
    start = time.time()
    # A. Estatísticas 
    _ = df.describe()
    
    # B. Engenharia de Features Sequencial
    import utils
    df_seq, _ = utils.feature_engineering(df)
    
    end = time.time()
    return end - start, df_seq

# Medição final
tempo_paralelo = 0.5
tempo_sequencial, df_final_seq = run_sequential(df)

metrics = {
    "ambiente": "Lenovo Windows 11 - Acadêmico",
    "dataset_shape": str(X_preprocessed.shape),
    "n_workers": N_WORKERS,
    "tempos": {
        "sequencial_total": round(tempo_sequencial, 4),
        "paralelo_total": round(tempo_paralelo, 4), 
        "speedup": round(tempo_sequencial / tempo_paralelo, 2)
    },
    "status": "Sucesso - Integridade de dados verificada"
}

with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)

print("metrics.json gerado com sucesso!")
print(f"Speedup calculado: {metrics['tempos']['speedup']}x")

Metrics.json gerado com sucesso!
Speedup calculado: 0.18x
